# Setup

In **Google Colab**, run the setup cell below: it downloads the versioned repository, installs DaliScope, and enables custom widgets. No local Python or Conda installation is needed. Use a standard CPU runtime.

In **local Jupyter**, install DaliScope using the root README and select **Python (DaliScope)**. If you update packages in an existing local session, restart the kernel once.

Run cells in order. The analysis setup locates the repository and sets the working directory to `notebooks` for relative data paths. See the [Colab guide](https://github.com/Liisa-Holm-group/DaliScope/blob/v0.1.3/docs/colab.md) for saving outputs and restarting a cloud runtime.


In [ ]:
# In Colab this installs DaliScope automatically. Locally, follow the README first.
import sys
from pathlib import Path
import subprocess
import runpy

try:
    import google.colab
except ModuleNotFoundError as error:
    if error.name not in {"google", "google.colab"}:
        raise
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    DALISCOPE_RELEASE = "v0.1.3"
    colab_root = Path("/content/DaliScope")
    if not colab_root.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", DALISCOPE_RELEASE,
            "https://github.com/Liisa-Holm-group/DaliScope.git", str(colab_root),
        ], check=True)
    setup_path = colab_root / "scripts/colab_setup.py"
    if not setup_path.is_file():
        raise RuntimeError("This runtime contains an older checkout. Start a fresh runtime and run setup again.")
    setup = runpy.run_path(str(setup_path))
    setup["setup_colab"](colab_root, DALISCOPE_RELEASE)
else:
    print("Local Jupyter: using the installed DaliScope environment.")


In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "daliscope").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "daliscope").is_dir():
    raise RuntimeError("Start in the DaliScope root or notebooks directory")
os.chdir(ROOT / "notebooks")
import daliscope
print("DaliScope", daliscope.__version__)

In [ ]:
import numpy as np
import pandas as pd
import scipy
import os
import py3Dmol
import anywidget
from html import escape
from IPython.display import Markdown, display

import matplotlib as plt
# Tell Jupyter's inline backend not to crop/re-scale figures
%config InlineBackend.print_figure_kwargs = {'bbox_inches': None}
#plt.rcParams['figure.figsize'] = (5,3)
plt.rcParams['figure.dpi'] = 100
#import daliscope
import daliscope.core.project
from daliscope.optics.headings import heading_plain, heading_banner, heading_underlined
from daliscope.mechanics.metrics import interpret_hits
from daliscope.mechanics.wizards import domain_wizard
from daliscope.optics.matplot import plot_dali_hits
from daliscope.optics.py3Dmol_viewer import show_domain_gallery
from daliscope.widgets.plotly_explorer import plotly_explorer
from daliscope.optics.msa import msa
import daliscope.widgets.factory
import daliscope.analysis.wrappers
from daliscope.analysis.pfam_stats import pfam_distribution_analysis
import daliscope.optics.histograms
import daliscope.analysis.domain_comparison
import daliscope.analysis.signature
import daliscope.analysis.DomNetPipeline
import daliscope.optics.domnet_viewer
import daliscope.analysis.data_preview
import daliscope.analysis.occupancy
import daliscope.analysis.CH_cluster
import daliscope.widgets.pairwise_py3Dmol
import daliscope.analysis.communities
import daliscope.optics.modules_sunburst

import seaborn as sns

import daliscope.widgets.domain_clipping

import ipywidgets as widgets
from IPython.display import display, HTML
from daliscope.widgets.plane_bisector import plane_bisector
from daliscope.analysis.data_preview import show_heading
from daliscope.core.project import validate_pipeline_state

from scipy.spatial import cKDTree
from typing import Dict, List


In [ ]:
%matplotlib inline
import plotly.io as pio
# Use the frontend-specific renderer for ordinary Plotly figures.
pio.renderers.default = "colab" if IN_COLAB else "notebook_connected"

import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# initialize
project = None

<a id="workflow-overview"></a>
<div style="background-color: #2b5c8f; color: white; padding: 15px; border-radius: 5px; margin-top: 40px;">

# User input - required

</div>
<br>
Replace data/test.tar.gz with the path to your uploaded data pack:
<p></p>

In [ ]:
pack_path = 'data/3ubpC_PDB25.tar.gz' 

Data packs can be supplied by your DALI workflow or generated with `daliscope-pack` when the required local protein and Pfam databases are available. See [data preparation](../docs/prepare-data.md).

Execute this comprehensive workflow section by section. Domain selection, optional clustering, and sequence signatures use explicit user inputs. The short `01_quickstart.ipynb` is the fresh-kernel entry example.

<div style="background-color: #2b5c8f; color: white; padding: 15px; border-radius: 5px; margin-top: 40px;">
    
# Run pipeline

</div>
The analysis pipeline consists of several sections:

1. [What is the global hit landscape?](#section-1) - Hit Density, Distance Falloff & Broad Quality Filters
2. [Which domain folds make up your protein?](#section-2) - Structural Coherence & Weighted Superimposition
3. [How do domain classifications characterize a population?](#section-3) - Pfam Family/Clan Inventory
4. [What subpopulations exist within the data?](#section-4) - Interactive view explorer & Custom view creation
5. [Are there hidden structural clusters in fold space?](#section-5) - Community detection in structural similarity network
6. [What is the structural context of a population?](#section-6) - Flanking Modular Architectures & Domain Plasticity
7. [What sequence and structural features characterize a population?](#section-7) - Multiple Structure Alignments viewed as Sequence Logos or heatmaps
8. [Does your protein contain an active or binding site?](#section-8) - Information Content (IC) Profiling & Catalytic Signatures
9. [Which specific proteins make up a population?](#section-9) - 3D Superimposition Viewer & Target Metadata Inspection
10. [How does your subset compare to a reference?](#section-10) - Hit landscape with Venn diagram

<hr style="border: 2px solid #ccc;">

# <span style="color: #2b5c8f;">Secction 1: What is the global hit landscape?</span>

<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    Hit Density, Distance Falloff & Broad Quality Filters
</div>

---

Once the raw Dali data are loaded, DaliScope presents the search results as a scatterplot of **Z-score versus query coverage**. Together, these provide a first view of the quality and extent of each structural match.

The two measures are generally correlated. The Dali Z-score summarizes structural similarity based on the correspondence of intramolecular distances: for a given number of aligned residues, a more precise structural match produces a higher Z-score. 

The scatterplot uses two complementary colour schemes:

* **Sequence identity** indicates how the structural matches relate to what could be found by sequence-based searches: from clear sequence homologs, through the *twilight zone* where structural alignment becomes particularly informative, to cases where structural similarity is difficult to detect from sequence alone.
* **Z-score regimes** provide a heuristic indication of the strength of the structural match. These regimes work well for typical cases, but their interpretation depends on the query structure: optimal thresholds can vary with protein size and shape.

As a first step, this view gives us a map of the structural neighbourhood of the Query and helps identify which hits warrant closer inspection. 
* DaliScope automatically creates two subsets wihch bisect the plane at valleys in the density distribution. Sometimes - but far from always - the subsets correspond to families or superfamilies. These automatically generated subsets are called `FILTERED_0` and `FILTERED_1`.

In [ ]:
# 1. Load data
project, FULL_DF, FULL_VIEW = daliscope.analysis.data_preview.load_and_inspect_project(pack_path)

# 2. Query metadata
show_heading("Query information")
project.show_query_meta()

# 3. Plot Landscape
fig = daliscope.analysis.data_preview.plot_dali_hits_with_boundaries(
    FULL_DF, 
    daliscope.analysis.data_preview.build_and_register_subsets(project, FULL_DF, FULL_VIEW)
)

# 4. Generate Dataset Statistics
print("Computing profiles...")
df_dict = {
    "FULL_DF": project.views[FULL_VIEW],
    "FILTERED_0": project.views["FILTERED_0"],
    "FILTERED_1": project.views["FILTERED_1"],
}
preview_summary_df = daliscope.analysis.data_preview.generate_summary_table(df_dict)

#5. Highlight table
show_heading("\nData set statistics")
print(
    "High fractions of twilight, dark or unannotated hits, and strong sequence\n"
    "conservation within a subset are highlighted.\n"
)
display(daliscope.analysis.data_preview.apply_summary_highlights(preview_summary_df))

# 5. Domain architecture hints
show_heading("Structural coverage")
daliscope.analysis.data_preview.composite_domain_coverage(
    FULL_DF, project.query_length
)

<hr style="border: 2px solid #ccc;">
<a id="section-2"></a>

# <span style="color: #2b5c8f;">Section 2: Which domain folds make up your protein?</span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    Structural Coherence & Weighted Superimposition
</div>

---
*([Back to Top](#workflow-overview))*

_**Skip this section if your Query structure is a single domain!**_

Dali alignments optimize intramolecular distance similarity rather than spatial coordinates. Distant aligned $\text{C}_\alpha$ atoms can therefore disrupt a rigid-body superimposition. This workflow uses two separate steps:
* **Domain clipping:** Fit the retained aligned residues by uniform least squares and recompute alignment metrics and domain annotations.
* **Occupancy-based filtering:** Score structural coverage using the population's occupancy profile and retain targets above the selected cutoff. This step inherits the clipped fit; it does not reweight or refit the superimposition.

Run **Cell I** to edit and visualize domain boundaries interactively.
> **Domain syntax uses one-based, inclusive PDB residue numbers:**
> - Continuous domain: `1-100`
> - Distinct domains: `1-100, 101-200`
> - Discontinuous domain: `201-300_400-500`

Run **Cell II** to register the domains. Custom overrides use the same PDB convention as the widget. The cell converts these ranges to the clipping API's zero-based, half-open convention, preserving the displayed residues.

Each domain yields a clipped view `dom_{i}` and an occupancy-filtered subset `dom_{i}_FOLD`. Domain boundaries and the occupancy cutoff are analysis choices; inspect their effects for your query.

Run **Cell III** to display the registered domain views.

In [ ]:
## Cell I: Interactive domain boundary widget

# Prepare PDB structure string
pdb_string = daliscope.optics.py3Dmol_viewer.xyz_to_ca_pdb(
    project.query_ca_coords, project.query_sequence
)

# Initialize and render interactive viewer
viewer = daliscope.optics.domnet_viewer.DomNetViewer(
    pdb_text=pdb_string,
    domain_string=project.puu_domains,
    normal_radius=0.65,
    highlight_radius=0.60,
    # segment_mask=segment_mask,
)

print("""
If the viewer throws a JavaScript error, refresh your browser (CTRL-R) and rerun this code cell.

Type one-based inclusive PDB domain ranges in the text box. When satisfied, run the next cell to register the domains. 
(The default is a size selected suggestion based on Dali's PUU (Protein Unfolding Units) hierarchy and it makes mistakes.)
""")
viewer.display()

In [ ]:
# Cell II: Register domains

# 0. Optional custom overrides use one-based inclusive PDB residues.
from daliscope.mechanics.domain_ranges import pdb_domain_string_to_clipping
custom_domain_string = ""
domain_string = (pdb_domain_string_to_clipping(custom_domain_string)
                 if custom_domain_string else viewer.clipping_domain_string)

custom_series_name = ""
series_name = custom_series_name if custom_series_name else "dom"

# 1. Register clipped parent domain. Updates superimpositions, alignment metrics (except Z-score which stays global) and Pfam annoations 
daliscope.mechanics.metrics.register_multiple_domains(
    domain_string, 
    project, 
    custom_name=series_name
)

# 2. Register filtered _FOLD subsets
show_heading("Create _FOLD subsets of clipped domains")
# 2.1. Find domains
df = project.list_inventory()
view_list = df.loc[df["function"] == "register_domain_view", "name"].tolist()

# 2.2. Process each registered domain view
for view_name in view_list:
    if view_name.endswith("_FOLD"): continue
    show_heading(view_name)
    daliscope.analysis.occupancy.create_fold_subset(project, view_name, show_plot=True, cutoff=0.5)

# 3. Render set overlap table
show_heading("Which parts of the protein correspond to distinct structural folds?")
shared_matrix = daliscope.analysis.occupancy.get_fold_matrix(project, view_list)
daliscope.analysis.occupancy.plot_shared_matrix_heatmap(shared_matrix, figsize=(3,2))

In [ ]:
# Cell III
items = [
    [project.provenance[name].parameters.get('domain_ranges'), name]
    for name in project.views.keys()
    if project.provenance.get(name) is not None
]

show_heading("Gallery of available domain views")
daliscope.optics.py3Dmol_viewer.show_domain_gallery(items, project.query_pdb_str, ncols=3)

<hr style="border: 2px solid #ccc;">
<a id="section-3"></a>

# <span style="color: #2b5c8f;">Section 3: How do domain classifications characterize a population?</span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    Pfam Family/Clan Inventory
</div>

---
*([Back to Top](#workflow-overview))*

Here, we show the Pfam family and superfamily (clan) assignments of the domain-fold populations. The table links family and clan identifiers to InterPro documentation. 

In [ ]:
validate_pipeline_state(glbs=globals()) # Ensures project and FULL_DF are completely safe

# pfam_counts in widget
daliscope.widgets.factory.launch_interactive_view(
    func=daliscope.analysis.wrappers.make_pfam_counts_table,
    title="Pfam counts",
    params={
        'k': {
            'type': 'intslider',
            'value': 10,
            'min': 1,
            'max': 5,
            'label': 'Show Top-k',
        }
    },
    project=project,
)

<hr style="border: 2px solid #ccc;">
<a id="section-4"></a>

# <span style="color: #2b5c8f;">Section 4: What subpopulations exist within the data?</span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    	Interactive view explorer & Custom view creation
</div>

---
*([Back to Top](#workflow-overview))*

## 4.1 Interactive scatterplot with hover text
Select a view of the data for exploratory data analysis.

- Run the cell twice if canvas is blank (it's due to canvas rendering synchronization issue).

- You can use Plotly's native Snapshot utility to save high-resolution figure as png. The menu icons show on hover in the toppom right corner of the canvas.

In [ ]:
validate_pipeline_state(glbs=globals()) # Ensures project and FULL_DF are completely safe

daliscope.widgets.plotly_explorer.launch_interactive_selectable_scatter(project, default_view_name=FULL_VIEW)

<hr style="border: 2px solid #ccc;">

## <span style="color: #2b5c8f;">ALTERNATIVE 4.1 (for Colab): Filtering based on alignment metrics (user input)</span>
---
<a id="section-3"></a>
*([Back to Top](#workflow-overview))*

Structural search results often contain both close homologs and more distant structural matches. To isolate the proteins most relevant to a particular analysis, hits can be filtered using structural alignment metrics, primarily **Z-score** and **alignment length**.

The **Z-score** measures the statistical significance of the structural alignment: higher values indicate that the observed similarity is less likely to occur by chance. **Alignment length** measures how much of the protein structure participates in the alignment. Considering both metrics together helps distinguish globally similar proteins from matches that share only a conserved structural motif or domain. For example, a hit with a high Z-score but a short alignment may represent a strong local similarity rather than an overall structural match.

The workflow consists of two steps:

#### Cell I — Set the Selection Boundary
Select the parent view containing the hits to filter. Position the bisecting line by dragging its anchor point and adjusting its angle. The selected side of the line defines the subset of hits to retain based on the desired combination of alignment metrics.

#### Cell II — Register the Subset
Confirm the selection and register it as a new subset view.

> **Note:** Advanced Pandas users can bypass the interactive widgets and call `project.add_subset()` directly with a custom `parent_view` and Boolean mask.


In [ ]:
# Cell I: Set boundary
%matplotlib widget

show_heading("Plane bisector widget")

print("Rerun this cell if the canvas is blank")
bisector = plane_bisector(project, df=None, k=10)

print("Run next cell to lock the selection")


In [ ]:
# Cell II: Lock and register

show_heading("Locking selection")

# 1. Harvest selection
CHILD_DF, params_dict = bisector.get_selected()
PARENT_VIEW = params_dict["view_name"]
PARENT_DF = project.views[PARENT_VIEW]
print(f"Harvested {len(CHILD_DF)} / {len(PARENT_DF)} targets in {PARENT_VIEW} based on active widget layout.")

# 2. Configure name (set custom_name to override)
custom_name = None
view_name = custom_name or daliscope.analysis.wrappers.next_subset_name(project, PARENT_VIEW, 'filtered')

# 3. Register subset
show_heading("Registering new view")
mask = PARENT_DF["target_id"].isin(CHILD_DF["target_id"])

_ = project.add_subset(
    name=view_name,
    mask=mask,
    parent=PARENT_VIEW,
    function='plane_bisector',
    parameters=params_dict,
)

display(CHILD_DF[['target_id', 'z_score', 'alignment_length', 'sequence_identity', 'description']])

## 4.2 💡Power User Option: Custom Query Filter
 
 Leave `query = ""` empty to skip filtering and retain all current hits.

 **Examples:**
 - High stringency: `query = "z_score >= 10.0 and rmsd <= 3.5"`
 - Sequence identity + coverage: `query = "sequence_identity >= 0.30 and query_coverage >= 0.80"`
 - Pfam filtering: `query = "clan == 'CL0042' or pfam == 'PF00018'"`

In [ ]:
# Edit the expression below; leave it empty to keep existing views unchanged.
query = ""
subset_name = "CUSTOM_SUBSET"

# A non-empty valid query registers the selected rows as a new subset.
# An empty query performs no registration and returns an all-False mask.
mask = project.add_custom_subset(query_str=query, subset_name=subset_name, source_view=FULL_VIEW)

## 4.3 Quality Metric Distributions per Pfam Family and Clan

> Unassigned & Other should be shown alongside top-k!

In [ ]:
validate_pipeline_state(glbs=globals()) # Ensures project and FULL_DF are completely safe

# Power User Cockpit: Fully exposed and modifiable directly in the notebook cell
title  = "Pfam/Clan Rank Distribution Violins"
params = {
        'k': {
            'type': 'intslider',
            'value': 5,
            'min': 1,
            'max': 30,
            'label': 'Top-k Ranks',
        }
}
fixed = {
        "metrics": ['z_score', 'query_coverage', 'sequence_identity']
}

# The One-Liner Execution Engine
daliscope.widgets.factory.launch_interactive_violin_plot(
    daliscope.analysis.wrappers.make_violin_plot, title, project, params, fixed
)

<hr style="border: 2px solid #ccc;">
<a id="section-5"></a>

# <span style="color: #2b5c8f;">Section 5: Are there hidden structural clusters in fold space? </span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    	Community detection in structural similarity network
</div>

---

*([Back to Top](#workflow-overview))*

This section is OFF by default.

 💡 **Power User Option: Structural Community Analysis (Infomap)**
 
 Set `run_rigid_core_pipeline = True` to identify rigid structural core modules across structural hits.
 >
 > **View Selection:** Default view is the full Query structure. 
 >
 > **Domain Range:** Leave `domain_range = None` to auto-fetch boundary coordinates from inventory metadata.

In [ ]:
df = project.list_inventory()
FOLD_view_list = df.loc[df["function"] == "register_domain_view", "name"].tolist()
print(f"\nThe following domain fold views have been defined: {FOLD_view_list}\n")

In [ ]:
# Enable/Disable pipeline execution

run_rigid_core_pipeline = False

view_name = FOLD_view_list[0]

# Configuration (Set domain_range = None to auto-extract from inventory)
domain_range = None  # e.g., (23, 123) or None for auto-resolution
k_neighbors = 200

if run_rigid_core_pipeline:
    # 1. Verify view existence
    if view_name not in project.views:
        raise KeyError(f"View '{view_name}' is not registered in project.views.")

    # 2. Dynamic domain_range resolution from inventory
    if domain_range is None:
        try:
            # Fingerprint/community ranges are one-based inclusive; clipping provenance is zero-based half-open.
            from daliscope.mechanics.domain_ranges import clipping_ranges_to_pdb
            domain_range = clipping_ranges_to_pdb(project.provenance[view_name].parameters['domain_ranges'])
        except Exception:
            domain_range = None

    if domain_range is None:
        raise ValueError(
            f"Could not automatically resolve domain_range for view '{view_name}'. "
            "Please specify domain_range = (start, end) manually."
        )

    child_view_name = f"{view_name}_FOLD"
    print(f"🚀 Running Community Detection on '{child_view_name}' with domain_range={domain_range}...")

    # 3. Community Detection Pipeline
    modules_df, n_modules = daliscope.analysis.communities.run_community_detection(
        view_name=child_view_name,
        domain_range=domain_range,
        project=project,
        k_neighbors=k_neighbors,
        max_workers=1,
        verbose=True,
    )

    # 4. Sunburst Visualizations
    fig, axes, clan_colors = daliscope.optics.modules_sunburst.plot_module_sunbursts(
        modules_df,
        modules=range(1, n_modules + 1),
    )

    # 5. Pfam / Clan Cross-Tabulation
    if n_modules > 1:
        print("\n--- Superfamily (Clan) Cross-Tabulation ---")
        daliscope.analysis.communities.module_pfam_table(
            project, modules_df, row_group="clan", min_count=20
        )

        print("\n--- Family (Pfam) Cross-Tabulation ---")
        daliscope.analysis.communities.module_pfam_table(
            project, modules_df, row_group="pfam", min_count=20
        )
else:
    print("Pipeline is OFF.")

<hr style="border: 2px solid #ccc;">
<a id="section-6"></a>

# <span style="color: #2b5c8f;">Section 6: What is the structural context of a population?</span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    	Flanking Modular Architectures & Domain Plasticity
</div>

---

*([Back to Top](#workflow-overview))*

## 6.1 Domain Architecture Cartoons
- Dashed red boxes indicate the leftmost and rightmost span of the clipped alignment
- Coloured boxes show Pfam annoations

In [ ]:
validate_pipeline_state(glbs=globals()) # Ensures project and FULL_DF are completely safe

title="Domain Architectures"
params={
        'k': {
            'type': 'intslider',
            'value': 10,
            'min': 5,
            'max': 30,  # Dynamically extended range
            'label': 'Top-k Ranks',
        }
}

daliscope.widgets.factory.launch_interactive_architecture_plot( 
    daliscope.analysis.wrappers.make_top_architecture_cartoons, title, project, params=params, 
)


## 6.2 How much of the full query and target does the structural alignment cover?
- `query_coverage` is aligned length divided by the **full query sequence length**, including in a clipped domain view.
- `target_coverage` is aligned length divided by the full target sequence length.

A short clipped domain can consequently have low query coverage even when it is almost completely aligned. These fractions alone do not establish whether a protein is single-domain or multidomain. Inspect domain boundaries, alignment length, structures, and annotations together. The `_FOLD` selection uses its separate occupancy-based coverage score.

In [ ]:
validate_pipeline_state(glbs=globals()) # Ensures project and FULL_DF are completely safe

# Launch it dynamically
daliscope.analysis.domain_comparison.launch_domain_wizard_widget(project)

<hr style="border: 2px solid #ccc;">
<a id="section-7"></a>

# <span style="color: #2b5c8f;">Section 7: What sequence and structural features characterize a population?</span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    	Multiple Structure Alignments viewed as Sequence Logos or heatmaps.
</div>

---

*([Back to Top](#workflow-overview))*

## 7.1 Interactive MSTA viewer
- Use left/right sliders to zoom in to region of interest 
- Secondary structure (dssp) colors: blue for helix, red for strand, green for coil, white for unaligned
- Amino acid colors: by amino acid groups (e.g. green for hydrophobic)

In [ ]:
validate_pipeline_state(glbs=globals()) # Ensures project and FULL_DF are completely safe

daliscope.widgets.factory.launch_interactive_msa_plot("Sequence / DSSP Viewer", project)


## 7.2 💡Power User Option: Pretty static alignment 

 Leave `view_name = ""` empty to skip this plot.
 
This uses the pymsaviz module. pymsaviz is slow, therefore output size is limited to the top 50 targets.
> - **Full alignment:** Leave `left = None` and `right = None`
> - **Trim segment:** Pass residue boundaries, e.g., `left = 100`, `right = 250`

In [ ]:
from pymsaviz import MsaViz, get_msa_testdata

view_name = None # e.g. "FILTERED_1"

if view_name:
    # sort by Z-scores
    df = project.views[view_name].sort_values('z_score', ascending=False)
    # run pymsaviz
    fig = daliscope.optics.msa.show_domain_alignments(
        df=df, 
        seq_col="sequ_pileup",
        id_col="target_id", 
        left=None,
        right=None,
        max_rows=60, 
    )
else:
    print("No view_name provided - skipping.")

<hr style="border: 2px solid #ccc;">
<a id="section-8"></a>

# <span style="color: #2b5c8f;">Section 8: Does your protein contain an active or binding site?</span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    	Information Content (IC) Profiling & Catalytic Signatures
</div>

---

*([Back to Top](#workflow-overview))*

This section is OFF by default unless FILTERED_1 shows a strong IC signal.	

### Signature Motif Pipeline

Crafting an appropriate signature profile is an iterative art: it requires selecting a sufficiently conserved yet structurally diverse SEED set, along with expert user judgment to identify functionally relevant signature positions.

The pipeline guides you through four key steps:

#### **Cell I — Select a SEED Set** *(User Input Required)*

Curate a representative and structurally diverse SEED set of proteins to serve as the baseline for profile generation.

#### **Cell II — Build the Weight Matrix** *(User Input Required)*

Lock the SEED alignment, generate a structural sequence logo, and hand-select the most informative, functionally key positions to construct a Prosite-like weight matrix.

#### **Cell III — Score & Classify Targets** *(User Input Required)*

Re-score all candidate target sequences against your weight matrix. Fine-tune the scoring threshold while critically evaluating the functional coherence and annotation quality of the resulting hits.

#### **Cell IV — Register the MOTIF**

Finalize and register the curated set of high-confidence hits as a new, fully annotated **MOTIF** subset.

In [ ]:
# Cell I:
validate_pipeline_state(glbs=globals()) # Ensures project and FULL_DF are completely safe

signal_threshold = 2.5

run_signature_pipeline = float(preview_summary_df.iloc[11][2]) > signal_threshold

# test signal of 'FILTERED_1'
if run_signature_pipeline:
    show_heading("Select filtered domain view as seed set")
    # RadioButtons wants list of (label, value) pairs
    options = [ (f"{key} (n={len(project.views[key])})", key) for key in list(project.views.keys()) ]

    selector = widgets.RadioButtons(
        options=options,
        value='FILTERED_1', # options[-1][1],
        description="Select filtered domain view:"
    )

    display(selector)

    print("\nRun next cell to lock the selection.")
else:
    print("Pipeline is OFF.")

In [ ]:
# CELL II:
validate_pipeline_state(glbs=globals())  # Ensures project and FULL_DF are completely safe

if run_signature_pipeline:
    # get SEED_VIEW value from widget, allow override
    default_parent_view = selector.value
    custom_parent_view = ""
    SEED_VIEW = custom_parent_view or default_parent_view

    show_heading(f"Locked {SEED_VIEW} as profile seed")

    nr_threshold = 0.40
    top_ic = 20
    profile_figures_before = set(plt.get_fignums())
    SEED_DF, nr_df, hc_profile, logo, profile_model_name, report = daliscope.analysis.signature.derive_signature_profile(
        project, SEED_VIEW, nr_threshold=nr_threshold, top_ic=top_ic
    )

    # Keep plots from earlier cells available for interaction.
    for profile_figure in set(plt.get_fignums()) - profile_figures_before:
        plt.close(profile_figure)
    show_heading("Validation in 3D: Select positions to be used for scoring")
    selector_ui, get_selected_residues = daliscope.optics.py3Dmol_viewer.make_residue_selector_widget(
        df=report,
        query_pdb_str=project.query_pdb_str,
        property_dict=(logo._info_df.sum(axis=1) * 20).to_dict(),  # IC content per position, used for coloring
    )
    display(selector_ui)
else:
    print("Pipeline is OFF.")

In [ ]:
# Cell III: rescore + interactive threshold
from daliscope.analysis.signature import compute_score_threshold, run_profile_screening_pipeline

if run_signature_pipeline:
    important_residues = get_selected_residues()
    print("Using selected residues for signature matching:")
    print(important_residues)

    show_heading("Functional validation")
    score_col_name = 'profile_score'
    threshold_percentage = 0.50
    initial_threshold = compute_score_threshold(hc_profile, important_residues, threshold_percentage)
    #final_results, get_threshold_state = daliscope.analysis.signature.run_profile_screening_pipeline(
    final_results, get_threshold_state = run_profile_screening_pipeline(
        hc_profile=hc_profile, full_df=FULL_DF, trusted_df=SEED_DF,
        positions=important_residues, initial_threshold=initial_threshold,
    )
else:
    print("Pipeline is OFF.")

In [ ]:
# Cell IV: run AFTER adjusting the slider above

if run_signature_pipeline:
    score_threshold = get_threshold_state()['score_threshold']

    # Single source of truth for the cutoff — SAME threshold_percentage used everywhere below
    matching_targets = final_results[final_results['profile_score'] >= score_threshold]

    show_heading("Registering MOTIF subset")
    default_name = daliscope.analysis.wrappers.next_subset_name(project, SEED_VIEW, 'motif')
    custom_name = None
    final_view_name = custom_name if custom_name else default_name

    # motif set's parent is always FULL_VIEW
    mask = FULL_DF["target_id"].isin(matching_targets["target_id"])

    _ = project.add_subset(
        name       = final_view_name,
        mask       = mask,
        parent     = FULL_VIEW,
        function   = 'rescore_targets_safely',
        parameters = {
            "nr_threshold": nr_threshold,
            "selected_positions": important_residues,
            "profile": profile_model_name,
            "threshold_percent": threshold_percentage,
            "score_threshold": score_threshold,
        },
    )
    motif_df = project.views[final_view_name]
    #print(f"\nRegistered {final_view_name} ({len(motif_df)})")

    show_heading("Targets in MOTIF subset")
    display_cols = ['target_id', 'z_score', 'alignment_length','sequence_identity', 'description']
    display(motif_df[display_cols])
else:
    print("Pipeline is OFF.")

<hr style="border: 2px solid #ccc;">
<a id="section-9"></a>

# <span style="color: #2b5c8f;">Section 9: Which specific proteins make up a population?	</span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    	3D Superimposition Viewer & Target Metadata Inspection
</div>

---

*([Back to Top](#workflow-overview))*

## 9.1 Pairwise Query - Target superimposition

In [ ]:
# Hand off to master launcher—it patches in view_name automatically
fixed = {
    'q_ca_coords': project.query_ca_coords,
    'target_ca_coords': project.coords,
}

daliscope.widgets.factory.launch_interactive_view(
        func=daliscope.widgets.pairwise_py3Dmol.create_superimposition_widget,
        title="Pairwise superimposition widget",
        params=None,
        fixed=fixed,
        project=project
)
    


## 9.2 Large table viewer

In [ ]:
# Power User Cockpit: Fully exposed and modifiable directly in the notebook cell
title = "Long Dataframe Window Viewer"

params = {
    'start_row': {
        'type': 'intslider',
        'value': 0,
        'min': 0,
        'max': len(project.master_metadata),
        'step': 25,
        'label': 'Start Row',
    },
    'window': {
        'type': 'intslider',
        'value': 5,
        'min': 5,
        'max': 100,
        'step': 5,
        'label': 'Rows per Page',
    }
}

# Configure default columns to display
fixed = {
    'columns': ['target_id', 'z_score', 'description', 'pfam', 'clan'],
    'max_cols': None
}

# The One-Liner Execution Engine
daliscope.widgets.factory.launch_interactive_table_viewer(
    daliscope.analysis.wrappers.make_table_window_view,
    title,
    project,
    params,
    fixed
)

<hr style="border: 2px solid #ccc;">
<a id="section-10"></a>

# <span style="color: #2b5c8f;">Section 10: How does your subset compare to a reference?</span>
<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    	Where do the domain populations fall in the hit landscape?
</div>

---

*([Back to Top](#workflow-overview))*

In [ ]:
project.list_inventory()

In [ ]:
# edit values to override defaults
df1 = FULL_DF 
comparison_view = (final_view_name if run_signature_pipeline else 'FILTERED_1')
df2 = project.views[comparison_view] # replace comparison_view with another registered name
df3 = None # e.g. project.views['FILTERED_1']
labels = ['FULL', comparison_view, 'FILTERED_1']

if df3 is not None and df2 is not None and df1 is not None:
  daliscope.optics.histograms.plot_3way_bright_comparison(
    df1, 
    df2, 
    df3, 
    on_cat='pfam', 
    x_var='z_score', 
    y_var='alignment_length', 
    labels=labels, 
    plot_KDE=True,
  )
elif df2 is not None and df1 is not None:
  daliscope.optics.histograms.plot_safe_comparison_with_marginal(
    df1, 
    df2, 
    on_category='pfam', 
    x_var='z_score', 
    y_var='alignment_length', 
    labels=labels, 
    max_points=15000,
  )
else:
    print("Nothing to plot")